In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
)
from imblearn.over_sampling import SMOTE

# Add project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features

# Load enriched dataset
df_feats, feature_cols = get_features("../data/raw")

# Filter seasons and minutes
df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008) &
    (df_feats["season_end_year"] <= 2024) &
    (df_feats["minutes_played"] >= 100)
].copy()

df_ml.shape


(23442, 79)

In [2]:
df_train = df_ml[df_ml["season_end_year"] <= 2018].copy()
df_val   = df_ml[(df_ml["season_end_year"] >= 2019) &
                 (df_ml["season_end_year"] <= 2022)].copy()

print("Train:", df_train.shape)
print("Val:", df_val.shape)

X_train = df_train[feature_cols]
y_train = df_train["ballon_dor_winner"].astype(int)

X_val = df_val[feature_cols]
y_val = df_val["ballon_dor_winner"].astype(int)


Train: (14301, 79)
Val: (6038, 79)


In [3]:
print("Before SMOTE:", y_train.value_counts())

sm = SMOTE(k_neighbors=1, random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("After SMOTE:", y_train_res.value_counts())
X_train_res.shape


Before SMOTE: ballon_dor_winner
0    14290
1       11
Name: count, dtype: int64
After SMOTE: ballon_dor_winner
0    14290
1    14290
Name: count, dtype: int64


(28580, 72)

In [4]:
scaler = StandardScaler()

X_train_res_scaled = scaler.fit_transform(X_train_res)
X_val_scaled = scaler.transform(X_val)


In [5]:
rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_res_scaled, y_train_res)


,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [6]:
proba_val = rf.predict_proba(X_val_scaled)[:, 1]
pred_val = (proba_val >= 0.5).astype(int)

print("AUC:", roc_auc_score(y_val, proba_val))
print("F1:", f1_score(y_val, pred_val))
print("Recall:", recall_score(y_val, pred_val))
print("Precision:", precision_score(y_val, pred_val))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, pred_val))


AUC: 0.9996133664733499
F1: 0.5
Recall: 0.3333333333333333
Precision: 1.0

Confusion Matrix:
[[6035    0]
 [   2    1]]


In [7]:
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]

importance_df = pd.DataFrame({
    "feature": [feature_cols[i] for i in idx],
    "importance": importances[idx]
})

importance_df.head(20)


,feature,importance
0,g_per90,0.130010
1,goals,0.112743
2,ga_per90,0.088371
3,g_per90_z,0.067774
4,ga_per90_z,0.067591
5,goals_z,0.066698
6,matches_played,0.050713
7,a_per90,0.046789
8,a_per90_z,0.043475
9,g_per90_z_lag1,0.042220


In [ ]:
from joblib import dump
dump(
    {"model": rf, "scaler": scaler, "feature_cols": feature_cols},
    "rf_baseline.pkl"      # ← SE GUARDA EN LA MISMA CARPETA DEL NOTEBOOK
)
print("Modelo guardado.")



Modelo guardado.
